Classification of mitotic phases in histopathology data
===================================================

Authors : 'Elyesse and Alexandre'

Supervision : 'Thomas and Raphaël'

Mitotic activity is a central indicator of tumor proliferation and prognosis. The most relevant information used today by pathologists is the count of mitotic events in selected regions. Identifying precise counts manually at the level of the entire tissue section is not possible,
as in a single slide there are hundreds of thousands of individual cells. AI solutions to automate the detection and counting process have been proposed, and the CBIO has developed solutions that are currently state-of-the-art (1st and 2nd place in two recent competitions).

However, we believe that more information on cell division could be leveraged by not only identifying and counting dividing cells, but also by dertermining the precise mitotic phase. Such an analysis could point to specific mechanisms of deregulation, potentially linked to clinical outcomes.

In this project, we propose to train Neural Networks to identify mitotic phases in Whole Slide Images.

Setup & configuration
===================================================

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import zipfile
import os
from google.colab import files

# Path to the zip file.
zip_file_path = 'data.zip'

print(f"Please upload the file '{zip_file_path}'.")
uploaded = files.upload()

if zip_file_path not in uploaded:
    print("No 'data.zip' file uploaded. Extraction cannot proceed.")
else:
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            # Extract to the current directory
            zip_ref.extractall('.')
        print(f"'{zip_file_path}' successfully extracted to the current directory.")
    except Exception as e:
        print(f"An error occurred during extraction: {e}")

Imports
===================================================


In [ ]:
!pip install -q torchstain torchcam grad-cam

In [ ]:
# --- 1. Standard & System Imports ---
import os
import random
import time
import copy
from pathlib import Path

# --- 2. Data & Image Manipulation ---
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import seaborn as sns

# --- 3. PyTorch & Torchvision ---
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.transforms import v2, InterpolationMode

# --- 4. Specific Libraries (Histopathology & Explainability) ---
import torchstain          # Color normalization
from torchcam.methods import SmoothGradCAMpp # For final visualization
from torchcam.utils import overlay_mask

# --- 5. Sklearn (Metrics) ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.manifold import TSNE
from scipy.stats import entropy

# --- 6. Grad-CAM (XAI) ---
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# --- Graphic Configuration ---
sns.set_theme(style="whitegrid") # For more aesthetic plots
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
from src.config import CONFIG, ROOT_DIR, IS_ATYPICAL, seed_everything
from src.data_analysis import load_and_analyze_metadata, compare_experts_agreement
from src.dataset import MitosisDataset
from src.utils import get_transforms, prepare_data_loaders, imshow, visualize_model
from src.train import train_model, get_convnext_model
from src.evaluation import evaluate_expert_comparison, plot_soft_confusion_matrix, analyze_uncertainty, evaluate_by_groups, run_full_test_battery
from src.visualization import plot_array, visualize_tsne_dual_expert, visualize_upstream_downstream_comparison, visualize_explanation_simple

seed_everything(42)

In [ ]:
df_stats = load_and_analyze_metadata('data/AMI-BR.csv')

In [ ]:
compare_experts_agreement(df_stats)

The dataset we are going to deal with is that of AMI-Br.

Dataset comes with a `AMI-BR.csv` file with annotations which looks like this:

``` {.sh}
,slide,dataset,uid,x,y,expert1_label,expert2_label,expert1_atypical,expert2_atypical,expert3_atypical,majority_atypical
0,002.tiff,MIDOG21,6,4422,216,NMF prometaphase,NMF metaphase,False,False,False,False
1,002.tiff,MIDOG21,7,1867,1583,AMF segregation,AMF segregation,True,True,True,True
```

Let\'s take a single image name and its annotations from the CSV, in
this case row index number 65 for `MIDOG21_194.png` just as an example.


In [ ]:
mitosis_frame = pd.read_csv('data/AMI-BR.csv')
class_names = sorted(mitosis_frame['expert1_label'].unique())

n = 65
img_name = f"{mitosis_frame.iloc[n]['dataset']}_{mitosis_frame.iloc[n]['uid']}.png"

print('Image name: {}, expert1_label: {}, expert2_label: {}'.format(img_name, mitosis_frame.iloc[n]['expert1_label'], mitosis_frame.iloc[n]['expert2_label']))

Dataset class
=============


Let\'s instantiate this class and iterate through the data samples. We
will print the sizes of first 4 samples and show their landmarks.


In [ ]:
mitosis_dataset = MitosisDataset(mitosis_frame, root_dir=ROOT_DIR)

fig = plt.figure()

for i, (image, label) in enumerate(mitosis_dataset):
    # Fix: Convert the soft label tensor to a single integer class index
    # by finding the index of the maximum value and then getting its item value.
    print(i, image.size,
          mitosis_dataset.classes[torch.argmax(label).item()])

    ax = plt.subplot(1, 4, i + 1)
    plt.tight_layout()
    ax.set_title('Sample #{}'.format(i))
    ax.axis('off')
    plt.imshow(image)
    plt.pause(0.001)

    if i == 3:
        plt.show()
        break

Load Data
=========

We will use torchvision and torch.utils.data packages for loading the
data.

The problem we\'re going to solve is to train a model to classify
**mitotic phases**. We have about 3 720 training images.

In [ ]:
# 6. DATALOADER PREPARATION (SPLIT & TRANSFORM)
# Note: We reuse df_stats (loaded in step 3)

dataloaders, (train_df, val_df, test_df), data_transforms = prepare_data_loaders(
    full_df=df_stats,
    config=CONFIG,
    root_dir=ROOT_DIR,
    get_transforms_fn=get_transforms, # Pass the function reference (Dependency Injection)
    dataset_cls=MitosisDataset        # Pass the class reference
)

# Quick check to ensure the loader is populated
print(f"Check Train Loader: {len(dataloaders['train'])} batches")

train_loader = dataloaders['train']
val_loader = dataloaders['val']
test_loader = dataloaders['test']
train_dataset = train_loader.dataset
val_dataset = val_loader.dataset
test_dataset = test_loader.dataset

Visualize a few images
======================

Let\'s visualize a few training images so as to understand the data
augmentations.


In [ ]:
# Get a batch of training data
inputs, classes = next(iter(train_loader))

# Make a grid from batch
out = torchvision.utils.make_grid(inputs)

# Convert soft labels (float tensors) to integer class indices for display
# We use argmax to get the most prominent class from the soft label
imshow(out, title=[train_dataset.classes[x.item()] for x in torch.argmax(classes, dim=1)])

In [ ]:
# --- SETUP ---
model = get_convnext_model(num_classes=len(train_dataset.classes), mode=CONFIG['TRAIN'])
model = model.to(CONFIG['DEVICE']) # Ensure 'device' is defined (cuda/cpu)

# --- THE AVOIDED PITFALL ---
# We only pass to the optimizer the parameters that need to be learned
params_to_update = []
for name, param in model.named_parameters():
    if param.requires_grad == True:
        params_to_update.append(param)

print(f"Number of parameters to train : {len(params_to_update)}")

# The optimizer only works on the filtered list
optimizer = optim.AdamW(params_to_update, lr=CONFIG['LR'], weight_decay=0.05)
criterion = nn.CrossEntropyLoss()

Train and evaluate
==================

In [ ]:
model_ft, train_history_loss, val_history_loss = train_model(model, dataloaders, criterion, optimizer, num_epochs=25)
plt.figure(figsize=(10, 5))
plt.plot(train_history_loss, label='Training Loss')
plt.plot(val_history_loss, label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
visualize_model(model_ft)

In [ ]:
# --- HOW TO USE IT ---
# 1. Load your best trained model
model_ft.load_state_dict(torch.load('best_model.pth'))

# 2. Run evaluation (Ensure you pass the corresponding DataFrame!)
evaluate_expert_comparison(model_ft, val_loader, val_df, CONFIG['DEVICE'])

# Visualization of Neural Networks

This section aims at exploring different ways of visualizing Neural Networks:
- t-SNE of representations
- grad-CAM
- activation maximization

First, some preliminaries that facilitate plotting and data access on Google drive ... just execute !

In [ ]:
plt.rcParams['figure.figsize'] = (8.0, 8.0)
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Visualization of the encodings by t-SNE

We will first visualize the data.

In [ ]:
# --- Execution ---
fig = plt.figure(figsize=(12, 12))
# Use train_dataset or val_dataset depending on what you have loaded
plot_array(fig, val_dataset, samples_per_class=10)
plt.tight_layout()
plt.show()

Now, we will extract the features of a layer and visualize the distribution of the encodings with t-SNE.

In [ ]:
# --- USAGE ---
# Note: use val_loader (shuffle=False) and val_df
visualize_tsne_dual_expert(model_ft, val_loader, val_df, num_samples=2000)

**Assignment**: Visualize now the embeddings at the two fully connected layers (in a different cell). Do you observe differences (albeit subtle)? Imagine you would like to use the same representations in another project (same image size, but other classes). Which of the representations seems more useful? Why?

In [ ]:
# Usage
visualize_upstream_downstream_comparison(model_ft, val_loader, val_df, num_samples=2000)

**Assignment**: Visualize the tSNE plot of the untrained embeddings (model output before training). What do you observe?

In [ ]:
# Load the "Untrained" model (Default ImageNet weights)
# We use 'feature_extraction' mode to load the model without attaching an optimizer.
# This model has never seen your mitosis images.
print("Loading pre-trained model (ImageNet)...")
model_untrained = get_convnext_model(num_classes=len(train_dataset.classes), mode='feature_extraction')
model_untrained = model_untrained.to(CONFIG['DEVICE'])
model_untrained.eval()

# Reuse the existing dual-expert visualization function
visualize_tsne_dual_expert(model_untrained, val_loader, val_df, num_samples=2000)

# Classification activation maps (grad-CAM)

Classification activation maps provide certainly the most popular visualization methods for network inspection.

In [ ]:
torch.cuda.empty_cache()

labels_names = train_dataset.classes

Now, we load an image from the `AMI-Br` data base.

In [ ]:
filename = 'MIDOG21_194.png'
folder_name = 'data/patches/normal'

image = Image.open(os.path.join(folder_name, filename)).resize((224, 224))
plt.title('MIDOG21_194')
plt.imshow(image)

Now, we will predict the label of the image.

In [ ]:
preprocess = data_transforms['train']

img_prep = preprocess(image)
img_prep = img_prep.unsqueeze(0)

with torch.no_grad():
    img_prediction = model(img_prep.to(CONFIG['DEVICE']))

# Get probabilities by applying softmax to the output
probs = F.softmax(img_prediction, dim=1)

# Convert the probabilities to a numpy array and get the top 3 predictions
top_probs, top_indices = torch.topk(probs, 4, dim=1)
top_probs = top_probs.squeeze().cpu().numpy()
top_indices = top_indices.squeeze().cpu().numpy()

for i in range(3):
    label = labels_names[top_indices[i]]
    print(f"{label} ({top_indices[i]}): {top_probs[i] * 100:.2f}%")

# Get the solution index with the maximum probability
max_index = top_indices[0]
print('Predicted Index:', max_index)

We will use the [Grad-Cam++](https://arxiv.org/pdf/1710.11063) implementation from torch-cam https://github.com/frgfm/torch-cam

Note that the output of the model needs to be the the logits and not probabilities (default behaviour in torchvision models). The reason is that with `softmax` we cannot study the influence of neurons on the output, if the output depends on all classes (which is the case when we use `softmax`).

In [ ]:
# Immediate test
# Note: I added transform=inference_transforms because the model needs
# normalized tensors to make a correct prediction (and for .unsqueeze to work).
inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
vis_dataset = MitosisDataset(
    train_df,
    root_dir=ROOT_DIR,
    transform=inference_transforms
)

# Pick a random image to test
random_idx = random.randint(0, len(vis_dataset)-1)
visualize_explanation_simple(model_ft, vis_dataset, random_idx, device=CONFIG['DEVICE'])

In [ ]:
# Usage
plot_soft_confusion_matrix(model_ft, val_loader, CONFIG['DEVICE'], train_dataset.classes)

In [ ]:
# Usage
analyze_uncertainty(model_ft, val_loader, CONFIG['DEVICE'])

In [ ]:
# --- USAGE ---
evaluate_by_groups(model_ft, val_loader, val_df, CONFIG['DEVICE'])

In [ ]:
run_full_test_battery(model_ft, dataloaders, test_df, train_dataset.classes, CONFIG['DEVICE'])